# **Stock Agent**

This notebook demonstrates the end-to-end invocation of the Stock Agent deployed on AWS Bedrock AgentCore Runtime.

**Contents:**
1. Setup & Configuration
2. User Authentication (Cognito)
3. Agent Queries (5 required test cases)
4. Langfuse Trace Verification

## **1. Setup & Configuration**

In [1]:
# Install required dependencies (run once)
!pip install --no-cache-dir boto3 httpx sseclient-py langfuse

In [2]:
import json
import os
import time
from datetime import datetime
import sseclient
import boto3
import httpx
import base64

# Configuration — update these values from Terraform outputs
AWS_REGION = "us-east-1"
COGNITO_USER_POOL_ID = "us-east-1_bkKtKtiOS"    # From: terraform output cognito_user_pool_id
COGNITO_CLIENT_ID = "jbkb083llr7ig76nd4nd4e95t"           # From: terraform output cognito_user_pool_client_id
AGENTCORE_ENDPOINT = "arn:aws:bedrock-agentcore:us-east-1:326597016036:runtime/stock_agent_dev_runtime-whyPK9D0wg/runtime-endpoint/stock_agent_dev_endpoint"       # From: terraform output agentcore_endpoint_url

# Test user credentials (create via step 2 below)
TEST_USER_EMAIL = "kevinlopez.ind@gmail.com"
TEST_USER_PASSWORD = "TestP@ssw0rd!23"

# Langfuse credentials (for trace verification)
LANGFUSE_PUBLIC_KEY = "pk-lf-cf8e1ecf-d5d7-45a1-bea9-528f3fae1325"
LANGFUSE_SECRET_KEY = "sk-lf-758e2297-6859-4ad8-adda-a616af313290"
LANGFUSE_BASE_URL = "https://us.cloud.langfuse.com"

print(f"Region: {AWS_REGION}")
print(f"User Pool: {COGNITO_USER_POOL_ID}")
print(f"Client ID: {COGNITO_CLIENT_ID}")
print(f"Endpoint: {AGENTCORE_ENDPOINT}")

Region: us-east-1
User Pool: us-east-1_bkKtKtiOS
Client ID: jbkb083llr7ig76nd4nd4e95t
Endpoint: arn:aws:bedrock-agentcore:us-east-1:326597016036:runtime/stock_agent_dev_runtime-whyPK9D0wg/runtime-endpoint/stock_agent_dev_endpoint


## **2. User Authentication (Cognito)**

### **2a. Create a Test User**

In [3]:
cognito_client = boto3.client("cognito-idp", region_name=AWS_REGION)

# Create test user (admin action — requires IAM permissions)
try:
    response = cognito_client.admin_create_user(
        UserPoolId=COGNITO_USER_POOL_ID,
        Username=TEST_USER_EMAIL,
        UserAttributes=[
            {"Name": "email", "Value": TEST_USER_EMAIL},
            {"Name": "email_verified", "Value": "true"},
        ],
        TemporaryPassword=TEST_USER_PASSWORD,
        MessageAction="SUPPRESS",  # Don't send welcome email
    )
    print(f"User created: {TEST_USER_EMAIL}")
    print(f"Status: {response['User']['UserStatus']}")
except cognito_client.exceptions.UsernameExistsException:
    print(f"User already exists: {TEST_USER_EMAIL}")

User already exists: kevinlopez.ind@gmail.com


In [4]:
# Set permanent password (required after admin_create_user)
try:
    cognito_client.admin_set_user_password(
        UserPoolId=COGNITO_USER_POOL_ID,
        Username=TEST_USER_EMAIL,
        Password=TEST_USER_PASSWORD,
        Permanent=True,
    )
    print("Password set to permanent.")
except Exception as e:
    print(f"Note: {e}")

Password set to permanent.


### **2b. Authenticate and Obtain JWT Tokens**

In [5]:
# Authenticate with USER_PASSWORD_AUTH flow
auth_response = cognito_client.initiate_auth(
    ClientId=COGNITO_CLIENT_ID,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={
        "USERNAME": TEST_USER_EMAIL,
        "PASSWORD": TEST_USER_PASSWORD,
    },
)

tokens = auth_response["AuthenticationResult"]
access_token = tokens["AccessToken"]
id_token = tokens["IdToken"]
refresh_token = tokens["RefreshToken"]

print("Authentication successful!")
print(f"Access Token (first 50 chars): {access_token[:50]}...")
print(f"ID Token (first 50 chars):     {id_token[:50]}...")
print(f"Token Type: {tokens['TokenType']}")
print(f"Expires In: {tokens['ExpiresIn']} seconds")

Authentication successful!
Access Token (first 50 chars): eyJraWQiOiJkVTEzSm1JK0tFRFJIRVwvVG4yd0hhVEI3RlJySD...
ID Token (first 50 chars):     eyJraWQiOiJzU3RSckFPS0QwZHhvZ3MyWEIybGFWRzBZNzhSWl...
Token Type: Bearer
Expires In: 3600 seconds


In [6]:
# Decode and display JWT claims (without verification — just for display)

def decode_jwt_payload(token: str) -> dict:
    """Decode JWT payload (base64) without verification."""
    payload = token.split(".")[1]
    # Add padding
    payload += "=" * (4 - len(payload) % 4)
    decoded = base64.b64decode(payload)
    return json.loads(decoded)

claims = decode_jwt_payload(id_token)
print("\nID Token Claims:")
print(json.dumps(claims, indent=2, default=str))


ID Token Claims:
{
  "sub": "e4e8f448-e0d1-707d-c80f-0f40d9daf9e2",
  "email_verified": true,
  "iss": "https://cognito-idp.us-east-1.amazonaws.com/us-east-1_bkKtKtiOS",
  "cognito:username": "e4e8f448-e0d1-707d-c80f-0f40d9daf9e2",
  "origin_jti": "7257b773-422d-49fc-acc7-6ddd88b17159",
  "aud": "jbkb083llr7ig76nd4nd4e95t",
  "event_id": "58911f44-9861-4a9e-b906-631052c67044",
  "token_use": "id",
  "auth_time": 1775678165,
  "exp": 1775681765,
  "iat": 1775678165,
  "jti": "4cb6aa03-e0de-47a5-8360-edc26d9bbaa3",
  "email": "kevinlopez.ind@gmail.com"
}


## **3. Agent Queries**

### Helper Function for SSE Streaming

In [7]:
# Query official

def query_agent_official(prompt: str, access_token: str, session_id: str = "production-session"):
    runtime_arn = AGENTCORE_ENDPOINT.split("/runtime-endpoint")[0]
    import urllib.parse
    encoded_arn = urllib.parse.quote(runtime_arn, safe="")
    url = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations"
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream"
    }
    payload = {"query": prompt, "thread_id": session_id}
    
    # Text-only professional output
    with httpx.stream("POST", url, headers=headers, json=payload, timeout=60.0) as response:
        if response.status_code != 200:
            print(f"Error [{response.status_code}]: {response.read().decode()}")
            return
            
        for line in response.iter_lines():
            if line.startswith("data: "):
                try:
                    data = json.loads(line[6:])
                    content = data.get("content", "")
                    
                    # CLIENT-SIDE METADATA STRIPPING
                    if isinstance(content, list):
                        for block in content:
                            if isinstance(block, dict) and "text" in block:
                                print(block["text"], end="", flush=True)
                            elif isinstance(block, str):
                                print(block, end="", flush=True)
                    elif isinstance(content, dict) and "text" in content:
                        print(content["text"], end="", flush=True)
                    elif isinstance(content, str):
                        print(content, end="", flush=True)
                except: pass
    
    print("\n\n[Query Complete]")


### Query 1: Real-Time Stock Price

In [8]:
response_1 = query_agent_official("What were Amazon's net sales in 2023 according to their 10-K?", id_token)

<thinking>The user is asking for specific financial information from Amazon's 10-K report. I should use the "search_knowledge_base" tool to find the net sales figure for 2023.</thinking>
According to the 2023 10-K report, Amazon's net sales in 2023 were $574.8 billion. This figure includes net sales from North America, International, and AWS segments. Here is the breakdown:

- **North America**: $352.8 billion
- **International**: $131.2 billion
- **AWS**: $90.8 billion

The consolidated net sales for 2023 were $574.8 billion.

[Query Complete]


### Query 2: Historical Stock Prices (Q4)

In [9]:
response_2 = query_agent_official("What were the stock prices for Amazon in Q4 last year?", id_token)

<thinking>To answer this question, I need to use the `retrieve_historical_stock_price` tool to get the historical stock prices for Amazon (ticker: AMZN) for the fourth quarter of last year. Q4 corresponds to the period from October 1 to December 31.</thinking>
<thinking>The tool has provided the historical stock prices for Amazon (AMZN) for the last three months. However, the data does not cover the entire Q4 period of last year. I need to adjust the period to cover the entire Q4 (October 1, 2025, to December 31, 2025) to get the stock prices for Q4 last year.</thinking> Here are the stock prices for Amazon (AMZN) for Q4 last year (October 1, 2025, to December 31, 2025):

- **October 2025:**
  - Start: $217.36
  - End: $250.10

- **November 2025:**
  - Start: $250.38
  - End: $222.69

- **December 2025:**
  - Start: $231.21
  - End: $230.82

**Summary:**
- **Start of Q4:** $217.36
- **End of Q4:** $230.82
- **Price Change:** +$13.46 (5.73% increase)

These figures provide a clear pictu

### Query 3: Stock Performance vs Analyst Predictions

In [10]:
response_3 = query_agent_official("Compare Amazon's recent stock performance to what analysts predicted in their reports", id_token)

<thinking>To compare Amazon's recent stock performance to analyst predictions, I need to retrieve both the current stock price and relevant analyst predictions from the knowledge base. I'll start by getting the current stock price and then search the knowledge base for any analyst predictions or performance expectations mentioned in Amazon's recent financial documents.</thinking>
Based on the retrieved data, here is a comparison of Amazon's recent stock performance to what analysts predicted in their reports:

### Current Stock Performance
- **Current Price:** $221.11
- **Previous Close:** $213.77
- **Day High:** $226.00
- **Day Low:** $219.62
- **Volume:** 39,060,163
- **Market Cap:** $2.37 trillion
- **P/E Ratio:** 30.88
- **52-Week High:** $258.60
- **52-Week Low:** $165.29

### Analyst Predictions
According to the knowledge base, Amazon provided forward-looking statements and guidance on February 6, 2025, which included:

- **First Quarter 2025 Guidance:**
  - **Net Sales:** Expect

### Query 4: AMZN Current Price + AI Business Info

In [11]:
response_4 = query_agent_official("I'm researching AMZN give me the current price and any relevant information about their AI business", id_token)

<thinking>The user has asked for the current stock price of Amazon (AMZN) and information about their AI business. I will first retrieve the current stock price using the `retrieve_realtime_stock_price` tool. Then, I will search the knowledge base for information about Amazon's AI business.</thinking>
Here is the information you requested about Amazon (AMZN):

### Current Stock Price
- **Ticker:** AMZN
- **Current Price:** $221.04
- **Previous Close:** $213.77
- **Day High:** $226.00
- **Day Low:** $219.62
- **Volume:** 39,331,876
- **Market Cap:** $2.37 trillion
- **P/E Ratio:** 30.87
- **52-Week High:** $258.60
- **52-Week Low:** $165.29
- **Timestamp:** 2026-04-08T19:58:39.721360

### AI Business Information
Amazon is heavily investing in AI to reinvent customer experiences across various domains. Here are some key points:

- **Investment in AI:** Amazon is building over 1,000 GenAI applications across shopping, coding, personal assistants, streaming, advertising, healthcare, and mo

### Query 5: Office Space (Knowledge Base Only)

In [12]:
response_5 = query_agent_official("What is the total amount of office space Amazon owned in North America in 2024?", id_token)

<thinking>The user is asking for specific information about Amazon's office space in North America for 2024. This information is likely to be found in Amazon's 2024 Annual Report. I will use the "search_knowledge_base" tool to find this information.</thinking>
As of December 31, 2024, Amazon owned a total of **9,104,000 square feet** of office space in North America. This information is found in the "Item 2. Properties" section of the 2024 Annual Report.

[Query Complete]


## **4. Langfuse Trace Verification**

### **4a. Fetch Recent Traces via Langfuse API**

In [13]:
# Query Langfuse API for recent traces
langfuse_headers = {
    "Authorization": f"Basic {base64.b64encode(f'{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}'.encode()).decode()}",
    "Content-Type": "application/json",
}

traces_response = httpx.get(
    f"{LANGFUSE_BASE_URL}/api/public/traces",
    headers=langfuse_headers,
    params={"limit": 10, "orderBy": "timestamp.desc"},
    timeout=30,
)

if traces_response.status_code == 200:
    traces_data = traces_response.json()
    traces = traces_data.get("data", [])
    
    print(f"Found {len(traces)} recent traces:\n")
    print(f"{'Trace ID':<40} {'Name':<25} {'Status':<10} {'Latency':<10} {'Timestamp'}")
    print("-" * 120)
    
    for trace in traces:
        trace_id = trace.get("id", "N/A")
        name = trace.get("name", "N/A")
        status = trace.get("status", "N/A")
        latency = trace.get("latency", "N/A")
        timestamp = trace.get("timestamp", "N/A")
        
        if isinstance(latency, (int, float)):
            latency_str = f"{latency/1000:.1f}s"
        else:
            latency_str = str(latency)
            
        print(f"{trace_id:<40} {name:<25} {status:<10} {latency_str:<10} {timestamp}")
else:
    print(f"Failed to fetch traces: {traces_response.status_code}")
    print(traces_response.text)

Found 0 recent traces:

Trace ID                                 Name                      Status     Latency    Timestamp
------------------------------------------------------------------------------------------------------------------------


In [14]:
# Display a detailed view of the most recent trace
if traces:
    latest_trace_id = traces[0]["id"]
    
    detail_response = httpx.get(
        f"{LANGFUSE_BASE_URL}/api/public/traces/{latest_trace_id}",
        headers=langfuse_headers,
        timeout=30,
    )
    
    if detail_response.status_code == 200:
        detail = detail_response.json()
        print(f"\nTrace Detail: {latest_trace_id}")
        print(f"Name: {detail.get('name')}")
        print(f"User: {detail.get('userId')}")
        print(f"Session: {detail.get('sessionId')}")
        print(f"Status: {detail.get('status')}")
        print(f"Input: {json.dumps(detail.get('input', {}), indent=2)[:200]}")
        print(f"Output: {json.dumps(detail.get('output', {}), indent=2)[:200]}")
        
        observations = detail.get("observations", [])
        print(f"\nObservations ({len(observations)}):")
        for obs in observations:
            print(f"  - {obs.get('type', '?')}: {obs.get('name', '?')} (latency: {obs.get('latency', '?')}ms)")
    else:
        print(f"Failed: {detail_response.status_code}")

### **4b. Langfuse dashboard**

[Langfuse Dashboard](https://us.cloud.langfuse.com) to view:

1. **Trace list** — All 5 agent invocations with latency and status
2. **Trace detail** — Nested spans showing LLM calls, tool usage, and retriever operations
3. **Cost tracking** — Token usage and estimated cost per invocation




![Langfuse Traces](screenshots/langfuse_tracess.png)

![Langfuse Trace Detail](screenshots/langfuse_trace_details.png)

## **5. AWS evidences** 

### **5a. Amazon Bedrock Agentcore** 

- #### **Runtime**

![Agentcore Runtime](screenshots/bedrock_agentcore_rutime.png)

- #### **Observability**

![Agentcore Runtime](screenshots/bedrock_agentcore_observability.png)

- #### **Endpoints**

![Agentcore Runtime](screenshots/bedrock_agentcore_endpoints.png)

### **5b. Amazon ECR** 

- #### **Repository**

![Agentcore Runtime](screenshots/ecr_repository.png)

- #### **Images**

![Agentcore Runtime](screenshots/ecr_repository.png)

### **5c. Amazon Cognito** 

![Agentcore Runtime](screenshots/cognito_overview.png)

### **5d. Amazon Secret Manager** 

![Agentcore Runtime](screenshots/secret_manager.png)